# Build a SQLite database from mMARCO Vietnamese triples

This Colab notebook streams the `triples/train` split from Hugging Face into one SQLite file. It does not keep the downloaded parquet shards, which avoids duplicating the dataset on Colab disk.

The split has about 39.8 million rows. Run with a small `MAX_ROWS` first to validate the workflow; set it to `None` for the full dataset.

In [ ]:
%pip install -q "datasets>=3.0" "tqdm>=4.0"

import sqlite3
from pathlib import Path

from datasets import load_dataset
from tqdm.auto import tqdm

In [ ]:
DATASET_ID = "minhnguyent546/mmarco-vietnamese-split"
DATASET_CONFIG = "triples"
DATASET_SPLIT = "train"
DB_PATH = Path("/content/mmarco_vietnamese_triples.db")

BATCH_SIZE = 10_000
MAX_ROWS = None  # Use 100_000 for a quick test, or None for all rows.

triples = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split=DATASET_SPLIT,
    streaming=True,
)

first_row = next(iter(triples))
required_columns = {"query", "positive", "negative"}
missing_columns = required_columns.difference(first_row)
if missing_columns:
    raise ValueError(f"Unexpected triples schema; missing: {sorted(missing_columns)}")

print(f"Columns: {list(first_row)}")
print(f"Output: {DB_PATH}")

In [ ]:
connection = sqlite3.connect(DB_PATH)
connection.execute("PRAGMA journal_mode = WAL")
connection.execute("PRAGMA synchronous = NORMAL")
connection.execute("PRAGMA temp_store = MEMORY")
connection.execute("""
    CREATE TABLE IF NOT EXISTS general (
        data_id TEXT PRIMARY KEY,
        source TEXT,
        title TEXT,
        topic TEXT,
        anchor TEXT NOT NULL,
        positive TEXT NOT NULL,
        hard_negative TEXT
    )
""")
connection.commit()

insert_query = """
    INSERT OR IGNORE INTO general (
        data_id, source, title, topic, anchor, positive, hard_negative
    ) VALUES (?, ?, ?, ?, ?, ?, ?)
"""

batch = []
processed = 0

try:
    for row_index, row in enumerate(tqdm(triples, desc="Writing SQLite")):
        if MAX_ROWS is not None and row_index >= MAX_ROWS:
            break

        batch.append(
            (
                f"data_004_001_{row_index:010d}",
                "mMARCO Vietnamese",
                None,
                None,
                row["query"],
                row["positive"],
                row["negative"],
            )
        )

        if len(batch) == BATCH_SIZE:
            connection.executemany(insert_query, batch)
            connection.commit()
            processed += len(batch)
            batch.clear()

    if batch:
        connection.executemany(insert_query, batch)
        connection.commit()
        processed += len(batch)
finally:
    connection.execute("PRAGMA wal_checkpoint(TRUNCATE)")
    connection.execute("PRAGMA journal_mode = DELETE")
    connection.close()

print(f"Processed rows this run: {processed:,}")

In [ ]:
connection = sqlite3.connect(DB_PATH)
row_count = connection.execute("SELECT COUNT(*) FROM general").fetchone()[0]
integrity = connection.execute("PRAGMA integrity_check").fetchone()[0]
connection.close()

print(f"Rows in database: {row_count:,}")
print(f"Integrity check: {integrity}")
print(f"Database size: {DB_PATH.stat().st_size / 1024**3:.2f} GB")

from google.colab import files
files.download(str(DB_PATH))